In [1]:
import boto3
import boto3
import duckdb
import tempfile
from datetime import datetime
import json

## Problem Statement

The raw data is ingested from an external source and stored as Parquet files in the Bronze layer (Amazon S3). Since the source data is in JSON format, fields that are missing or empty in a particular batch are not included in the generated Parquet files. As a result, the schema of incoming Parquet files may vary across batches.

### Possible scenarios include:

- We are ingesting data from S3 where data is split across many Parquet files
- New columns being introduced in future batches.
- Existing columns being absent in older files.
- Each batch may have a different schema
  
If these files are appended directly into a single table, the ingestion pipeline becomes unreliable and can fail due to schema mismatches.

Therefore, instead of trusting every incoming file, we can introduce a **master schema** that acts as a single source of truth for all incoming data.


## Design Philosophy

The pipeline separates **storage**, **schema management**, and **computation** into independent responsibilities.

| Component | Responsibility |
|-----------|---------------|
| **Bronze (S3)** | Stores raw data exactly as received. No transformations are performed. |
| **Master Schema (JSON)** | Defines the expected schema for the Bronze staging table. Acts as the schema contract. |
| **DuckDB** | Temporary computation engine used to normalize and prepare data before writing it to the Silver layer. |

The objective is to ensure that **every dataset entering DuckDB follows the same structure**, regardless of the schema of the incoming Parquet file.



In [2]:
def extract_timestamp(folder_name):
    timestamp_str = folder_name.rstrip("/").replace("ingest_date_", "")
    return datetime.strptime(timestamp_str, "%Y%m%d_%H%M%S")

## Incremental Data Loading Strategy

The Bronze layer stores data as **time-partitioned Parquet files** in Amazon S3. Each ingestion creates a new folder with the following naming convention:

```text
bronze/full_load/
    ├── ingest_date_20260701_103000/
    ├── ingest_date_20260702_091030/
    ├── ingest_date_20260703_101500/
    └── .
```

Instead of scanning the entire Bronze layer during every pipeline execution, the ingestion process follows an **incremental loading strategy**.

A start timestamp and an end timestamp are provided as input to the pipeline. Using these timestamps, only the folders whose ingestion timestamp falls within the specified time window are selected for processing.

This approach ensures that:

- Only newly ingested data is processed.
- Previously processed batches are not scanned again.
- The amount of data read from Amazon S3 is significantly reduced.
- Pipeline execution time and compute cost remain low.


In [3]:
def load_valid_folders(start_date,end_date,bucket_name,prefix) :
    valid_folder = {}

    response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix = prefix ,
    Delimiter="/"
    )

    for folder in response.get("CommonPrefixes", []):
        folder_name = folder["Prefix"].split("/")[-2]

        folder = f"{folder_name}/"
        ts = extract_timestamp(folder)

        if ts >= start_date and ts <end_date:
            valid_folder[folder_name]=ts
            print(folder_name)

        else:
            pass

    return valid_folder
    

In [4]:
def load_master_schema():
    with open("master_schema.json", "r") as f:
        schema_dict = json.load(f)
    return schema_dict

def update_scheme_master(column_names, schema_dict):

    columns_added = 0 
    new_columns =[]
    for col in column_names :
        if col not in schema_dict :
            schema_dict[col] = "VARCHAR"
            print("New Column Found : ", col)
            columns_added+=1
            new_columns.append(col)

    if columns_added ==0 :
        pass
    else:
        try:
            json.dumps(schema_dict)
        
            with open("master_schema.json", "w") as f:
                json.dump(schema_dict, f, indent=4)

        except:
            print("Error")            

    return new_columns
    
#schema_dict= load_master_schema()
#update_scheme_master(column_names,schema_dict)
#schema_dict= load_master_schema()
#print(schema_dict)

## Schema Evolution Strategy

Instead of assuming that every incoming file has the correct schema, the pipeline follows a **schema evolution** approach.

For every incoming Parquet file:

1. Extract the column names.
2. Compare them with the existing `master_schema.json`.
3. If a new column is detected:
   - Add it to the master schema.
   - Use it for all future processing.
   - Create a flow to further backfill/handle data loaded in previous loads

This ensures that the schema continuously evolves without requiring manual intervention.

---

### Schema Alignment Logic

Once the master schema has been finalized, every incoming dataset is projected onto this schema.

If the column does **not** exist:
  - Insert `NULL` for that column.

Conceptually:

```
      This guarantees that every row entering DuckDB has an identical structure.
```


### Edge Cases Considered

#### 1. New Columns Appear

**Scenario**

The source application introduces additional fields.

- Detect the new column.
- Update the master schema.
- Include the column in future processing.

#### 2. Missing Columns

**Scenario**

Older Parquet files do not contain recently added columns.

**Handling**

- Insert `NULL`.
- Preserve the record.
- Maintain schema consistency.

---

#### 3. Schema Drift Across Files

**Scenario**

Different files inside the same ingestion batch have different schemas.

**Handling**

Each file is independently aligned with the master schema before insertion.

---

#### 4. 

**Scenario**

The same column appears as different types across files.

**Handling**

All columns are initially converted to `VARCHAR`.

Data type validation and conversion are deferred to the Silver layer.


In [5]:
def load_data_to_duckdb(valid_folder,Prefix):
    #connect to s3 bucket
    s3 = boto3.client("s3")
    bucket = "nyi-data-analytics"
    
    #connect to DuckDb
    con = duckdb.connect("bronze_staging_db.duckdb")

    #drop the old table (to fetch & clean only the new_data) & Creates One if First Run
    con.execute("DROP TABLE IF EXISTS staging_bronze;")

    files_pushed = 0

    for folder_name, ts in valid_folder.items():

        prefix = f"{Prefix}{folder_name}/"

        # list parquet files
        resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

        #we will traverse each parquet file , type cast it & upload to duckdb for transformations
        for obj in resp.get("Contents", []):
            key = obj["Key"]
    
            # download file
            file_obj = s3.get_object(Bucket=bucket, Key=key)

            #write file data on temp file
            with tempfile.NamedTemporaryFile(suffix=".parquet") as f:
                f.write(file_obj["Body"].read())
                f.flush()
    
                print("Fetched Data from location : ",key)

                #read columns to type cast them
                schema = con.execute("""
                    DESCRIBE SELECT * FROM read_parquet(?)
                    limit 0
                    """, [f.name]).fetchdf()
                
                column_names = schema["column_name"].tolist()

                schema_dict= load_master_schema()

                new_columns = update_scheme_master(column_names,schema_dict)

                if len(new_columns) == 0 :
                    pass
                else:
                    #we are adding these columns so that if we encounter new columns we can handle them by adding column in DB
                    #Edge Case : New coumn Introduced in the dataset
                    for col in new_columns:
                        con.execute(f"""
                        ALTER TABLE staging_bronze
                        ADD COLUMN "{col}" VARCHAR""")
                        print(f"Column {col} added into the table")

                schema_dict= load_master_schema()

                select_expr=[]
                for col in schema_dict.keys():
                    if col in column_names:
                        #column values are present
                        select_expr.append(f'CAST("{col}" AS VARCHAR) AS "{col}"')
                    else:
                        print(f"\t\t {col} not present in existing data , values pushed as null")
                        select_expr.append(f'NULL as "{col}"') 
                        
                con.execute(f"""
                CREATE TABLE IF NOT EXISTS staging_bronze AS
                SELECT {select_expr},
                       ? AS ingest_timestamp
                FROM read_parquet(?,union_by_name=true)
                limit 0
                """,[ts,f.name])
                
                con.execute(f"""
                INSERT INTO staging_bronze
                SELECT
                    {select_expr},
                    ? AS ingest_timestamp
                FROM read_parquet(?, union_by_name=true)
                """, [ts, f.name])

                files_pushed += 1

    print(files_pushed)
    return
                   

In [6]:
s3 = boto3.client("s3")

bucket_name = "nyi-data-analytics"
Prefix="bronze/full_load/"
#prefix ko abhi bronze/full_load karna hai

#the dates here are ingestion timestamp
start_date = datetime.strptime("2026-07-01 10:29:00", "%Y-%m-%d %H:%M:%S")
end_date = datetime.strptime("2026-07-03 00:00:00", "%Y-%m-%d %H:%M:%S")

#load valid folders (basically we want to fetch only the data that is required for staging not old data
valid_folder=load_valid_folders(start_date,end_date,bucket_name,Prefix)

load_data_to_duckdb(valid_folder,Prefix)

print()

ingest_date_20260702_091030
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_001.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_002.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_003.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_004.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_005.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_006.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_007.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_008.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_009.parquet
Fetched Data from location :  bronze/full_load/ingest_date_20260702_091030/part_010.parquet
Fetched Data from location :  bronze/full_load/inges

In [7]:
#Note the count of rows is exactly as same as what we fetched from in the S3 Load
con = duckdb.connect("bronze_staging_db.duckdb")
con.execute("select count(*) from staging_bronze").fetchall()

[(946320,)]